In [1]:
# BirdCLEF+ 2026 — EfficientNetV2-B2 on log-mel spectrograms

!pip install colorednoise --no-index --find-links=/kaggle/input/datasets/imokutmfonudoh/colorednoise --no-deps

import os, gc, math, random, warnings
warnings.filterwarnings("ignore")
os.environ["TF_USE_LEGACY_KERAS"] = "1"
from pathlib import Path
import numpy as np
import pandas as pd
import librosa, colorednoise as cn
from sklearn.model_selection import StratifiedKFold
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, mixed_precision


print(f"TensorFlow : {tf.__version__}")
print(f"GPUs found : {tf.config.list_physical_devices('GPU')}")

# 1. CONFIGURATION
class CFG:
    BASE_DIR               = Path("/kaggle/input/competitions/birdclef-2026/")
    TRAIN_AUDIO_DIR        = BASE_DIR / "train_audio"
    TRAIN_SOUNDSCAPES_DIR  = BASE_DIR / "train_soundscapes"
    TEST_SOUNDSCAPES_DIR   = BASE_DIR / "test_soundscapes"
    TRAIN_CSV              = BASE_DIR / "train.csv"
    TAXONOMY_CSV           = BASE_DIR / "taxonomy.csv"
    SOUNDSCAPE_LABELS_CSV  = BASE_DIR / "train_soundscapes_labels.csv"
    SAMPLE_SUB_CSV         = BASE_DIR / "sample_submission.csv"

    SAMPLE_RATE = 32_000
    DURATION    = 5
    N_SAMPLES   = SAMPLE_RATE * DURATION

    N_FFT      = 1024
    HOP_LENGTH = 320
    N_MELS     = 128
    FMIN       = 20
    FMAX       = 16000

    IMG_HEIGHT = 128
    IMG_WIDTH  = 384
    IMG_SHAPE  = (IMG_HEIGHT, IMG_WIDTH, 3)

    BACKBONE   = "EfficientNetV2B2"
    DROP_RATE  = 0.3
    DROP_RATE2 = 0.2

    BATCH_SIZE    = 32
    EPOCHS        = 2
    LEARNING_RATE = 1e-3
    MIN_LR        = 1e-6
    WEIGHT_DECAY  = 1e-4
    WARMUP_EPOCHS = 3
    PATIENCE      = 7
    N_FOLDS       = 5
    TRAIN_FOLDS   = [0]
    SEED          = 42

    USE_MIXUP        = True
    MIXUP_ALPHA      = 0.4
    USE_SPEC_AUGMENT = True
    FREQ_MASK_MAX    = 20
    TIME_MASK_MAX    = 40
    USE_NOISE        = True
    NOISE_PROB       = 0.5
    SNR_DB_MIN       = 5
    SNR_DB_MAX       = 30

    SECONDARY_LABEL_WEIGHT = 0.5
    MIN_RATING             = 0.0
    USE_MIXED_PRECISION    = True

    SOUNDSCAPE_DURATION = 60
    INFER_OVERLAP       = 0.0
    INFER_BATCH_SIZE    = 64

def set_seed(seed=CFG.SEED):
    random.seed(seed); np.random.seed(seed); tf.random.set_seed(seed)

set_seed()

if CFG.USE_MIXED_PRECISION:
    mixed_precision.set_global_policy("mixed_float16")
    print("Mixed precision: ON")

# 2. DATA LOADING & LABEL ENCODING
train_df   = pd.read_csv(CFG.TRAIN_CSV)
taxonomy   = pd.read_csv(CFG.TAXONOMY_CSV)
sample_sub = pd.read_csv(CFG.SAMPLE_SUB_CSV)
sc_labels  = pd.read_csv(CFG.SOUNDSCAPE_LABELS_CSV)

ALL_SPECIES    = [c for c in sample_sub.columns if c != "row_id"]
SPECIES_TO_IDX = {s: i for i, s in enumerate(ALL_SPECIES)}
NUM_CLASSES    = len(ALL_SPECIES)

print(f"Total species : {NUM_CLASSES} | Training files : {len(train_df)}")

train_df["filepath"] = train_df["filename"].apply(lambda x: str(CFG.TRAIN_AUDIO_DIR / x))

if CFG.MIN_RATING > 0:
    train_df = train_df[
        (train_df["rating"] >= CFG.MIN_RATING) | (train_df["rating"] == 0)
    ].reset_index(drop=True)

train_df = train_df[train_df["primary_label"].isin(SPECIES_TO_IDX)].reset_index(drop=True)
train_df["label_idx"] = train_df["primary_label"].map(SPECIES_TO_IDX)

print(f"After quality filter : {len(train_df)} samples")
print(train_df["primary_label"].value_counts().head())

# 3. SOUNDSCAPE LABELS → TRAINING ROWS
def timestamp_to_seconds(ts):
    try: return float(ts)
    except: pass
    parts = [float(p) for p in str(ts).strip().split(":")]
    if len(parts) == 3: return parts[0]*3600 + parts[1]*60 + parts[2]
    if len(parts) == 2: return parts[0]*60 + parts[1]
    return parts[0]

def parse_soundscape_labels(labels_df):
    rows = []
    for _, row in labels_df.iterrows():
        fp = str(CFG.TRAIN_SOUNDSCAPES_DIR / row["filename"])
        if not os.path.exists(fp): continue
        valid = [l.strip() for l in str(row["primary_label"]).split(";")
                 if l.strip() in SPECIES_TO_IDX]
        if not valid: continue
        rows.append({
            "filepath": fp, "primary_label": valid[0],
            "secondary_labels": ",".join(valid[1:]),
            "label_idx": SPECIES_TO_IDX[valid[0]],
            "start_sec": timestamp_to_seconds(row["start"]),
            "end_sec": timestamp_to_seconds(row["end"]),
            "rating": 5.0, "is_soundscape": True,
        })
    return pd.DataFrame(rows)

sc_train_df = parse_soundscape_labels(sc_labels)
print(f"Soundscape segments : {len(sc_train_df)}")

# 4. CROSS-VALIDATION SPLIT
skf = StratifiedKFold(n_splits=CFG.N_FOLDS, shuffle=True, random_state=CFG.SEED)
train_df["fold"] = -1
for fold, (_, val_idx) in enumerate(skf.split(train_df, train_df["label_idx"])):
    train_df.loc[val_idx, "fold"] = fold
print(train_df["fold"].value_counts().sort_index())

# 5. AUDIO PREPROCESSING
def _make_mel_filterbank():
    return tf.signal.linear_to_mel_weight_matrix(
        num_mel_bins=CFG.N_MELS, num_spectrogram_bins=CFG.N_FFT//2+1,
        sample_rate=CFG.SAMPLE_RATE, lower_edge_hertz=CFG.FMIN,
        upper_edge_hertz=CFG.FMAX, dtype=tf.float32,
    )

MEL_FILTERBANK = _make_mel_filterbank()

def load_audio(filepath, sr=CFG.SAMPLE_RATE, offset=0.0, duration=None):
    try:
        audio, _ = librosa.load(filepath, sr=sr, offset=offset, duration=duration, mono=True)
        return audio.astype(np.float32)
    except:
        return np.zeros(CFG.N_SAMPLES, dtype=np.float32)

def load_and_pad_cpu(filepath, start_sec=-1.0, training=False):
    offset = float(start_sec) if start_sec >= 0 else 0.0
    audio  = load_audio(filepath, offset=offset, duration=CFG.DURATION if start_sec >= 0 else None)

    if training and CFG.USE_NOISE and np.random.random() < CFG.NOISE_PROB:
        noise = cn.powerlaw_psd_gaussian(1, len(audio)).astype(np.float32)
        snr   = np.random.uniform(CFG.SNR_DB_MIN, CFG.SNR_DB_MAX)
        sig_p = np.mean(audio**2) + 1e-9
        noi_p = np.mean(noise**2) + 1e-9
        noise *= np.sqrt((sig_p / (10 ** (snr / 10.0))) / noi_p)
        audio += noise

    n = CFG.N_SAMPLES
    if len(audio) == 0: return np.zeros(n, dtype=np.float32)
    if len(audio) >= n:
        start = np.random.randint(0, len(audio) - n + 1)
        return audio[start:start+n]
    return np.tile(audio, math.ceil(n / len(audio)))[:n]

@tf.function(jit_compile=False)
def waveform_to_image(waveform):
    stft = tf.signal.stft(waveform, frame_length=CFG.N_FFT, frame_step=CFG.HOP_LENGTH,
                          fft_length=CFG.N_FFT, window_fn=tf.signal.hann_window, pad_end=True)
    power = tf.math.real(stft * tf.math.conj(stft))
    mel   = tf.tensordot(power, MEL_FILTERBANK, axes=[[1], [0]])
    mel.set_shape([None, CFG.N_MELS])
    log_mel = tf.math.log(mel + 1e-6)
    log_mel = (log_mel - tf.reduce_min(log_mel)) / (tf.reduce_max(log_mel) - tf.reduce_min(log_mel) + 1e-8)
    log_mel = tf.transpose(log_mel)
    img = tf.image.resize(tf.expand_dims(log_mel, axis=-1), [CFG.IMG_HEIGHT, CFG.IMG_WIDTH])
    return tf.concat([img, img, img], axis=-1)

def process_sample(filepath, start_sec=-1.0, training=False):
    audio = load_and_pad_cpu(filepath, start_sec=start_sec, training=training)
    return waveform_to_image(tf.constant(audio)).numpy()

# 6. AUGMENTATION
@tf.function
def spec_augment(image):
    H, W, C = CFG.IMG_HEIGHT, CFG.IMG_WIDTH, 3
    f  = tf.random.uniform((), 0, CFG.FREQ_MASK_MAX, dtype=tf.int32)
    f0 = tf.random.uniform((), 0, H - f, dtype=tf.int32)
    image *= tf.concat([tf.ones([f0,W,C],tf.float32), tf.zeros([f,W,C],tf.float32),
                        tf.ones([H-f0-f,W,C],tf.float32)], axis=0)
    t  = tf.random.uniform((), 0, CFG.TIME_MASK_MAX, dtype=tf.int32)
    t0 = tf.random.uniform((), 0, W - t, dtype=tf.int32)
    image *= tf.concat([tf.ones([H,t0,C],tf.float32), tf.zeros([H,t,C],tf.float32),
                        tf.ones([H,W-t0-t,C],tf.float32)], axis=1)
    return image

@tf.function
def apply_mixup(images, labels):
    B   = tf.shape(images)[0]
    lam = tf.maximum(tf.random.stateless_uniform([B], seed=[CFG.SEED,0]), 0.5)
    idx = tf.random.shuffle(tf.range(B))
    lam_img = tf.reshape(tf.cast(lam, tf.float32), [B,1,1,1])
    lam_lbl = tf.reshape(tf.cast(lam, tf.float32), [B,1])
    mixed_images = lam_img * images + (1-lam_img) * tf.gather(images, idx)
    mixed_labels = lam_lbl * labels + (1-lam_lbl) * tf.gather(labels, idx)
    return mixed_images, mixed_labels

# 7. TF.DATA PIPELINE
def make_target_vector(primary_label, secondary_labels_str=""):
    target = np.zeros(NUM_CLASSES, dtype=np.float32)
    if primary_label in SPECIES_TO_IDX:
        target[SPECIES_TO_IDX[primary_label]] = 1.0
    if secondary_labels_str:
        for s in str(secondary_labels_str).split(","):
            s = s.strip().strip("[]'\\\" ")
            if s in SPECIES_TO_IDX and target[SPECIES_TO_IDX[s]] == 0.0:
                target[SPECIES_TO_IDX[s]] = CFG.SECONDARY_LABEL_WEIGHT
    return target

def _py_load_waveform(filepath_b, primary_b, secondary_b, start_sec_f, training_b):
    waveform = load_and_pad_cpu(filepath_b.numpy().decode(), start_sec=float(start_sec_f.numpy()),
                                training=bool(training_b.numpy()))
    target   = make_target_vector(primary_b.numpy().decode(), secondary_b.numpy().decode())
    return waveform, target

@tf.function
def _gpu_process(waveform, target, training):
    img = waveform_to_image(waveform)
    if training and CFG.USE_SPEC_AUGMENT: img = spec_augment(img)
    return img, target

def build_dataset(df, training=False):
    filepaths   = df["filepath"].astype(str).values
    primaries   = df["primary_label"].astype(str).values
    secondaries = df.get("secondary_labels", pd.Series([""]*len(df))).fillna("").astype(str).values
    start_secs  = df.get("start_sec", pd.Series([-1.0]*len(df))).fillna(-1.0).astype(np.float32).values

    def tf_load_waveform(fp, prim, sec, start, train_f):
        waveform, target = tf.py_function(_py_load_waveform, [fp,prim,sec,start,train_f],
                                          [tf.float32, tf.float32])
        waveform.set_shape((CFG.N_SAMPLES,)); target.set_shape((NUM_CLASSES,))
        return waveform, target

    ds = tf.data.Dataset.from_tensor_slices(
        (filepaths, primaries, secondaries, start_secs, np.array([training]*len(df), dtype=bool))
    )
    if training: ds = ds.shuffle(min(len(df), 8192), seed=CFG.SEED, reshuffle_each_iteration=True)
    ds = ds.map(tf_load_waveform, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(CFG.BATCH_SIZE, drop_remainder=training)
    ds = ds.map(lambda wav, tgt: tf.vectorized_map(
                    lambda p: _gpu_process(p[0], p[1], training), (wav, tgt)),
                num_parallel_calls=tf.data.AUTOTUNE)
    if training and CFG.USE_MIXUP: ds = ds.map(apply_mixup, num_parallel_calls=tf.data.AUTOTUNE)
    return ds.prefetch(tf.data.AUTOTUNE)

Looking in links: /kaggle/input/datasets/imokutmfonudoh/colorednoise
Processing /kaggle/input/datasets/imokutmfonudoh/colorednoise/colorednoise-2.2.0-py3-none-any.whl


2026-04-22 14:21:35.562105: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776867695.744630      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776867695.798725      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776867696.248811      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776867696.248854      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776867696.248856      55 computation_placer.cc:177] computation placer alr

TensorFlow : 2.19.0
GPUs found : []
The dtype policy mixed_float16 may run slowly because this machine does not have a GPU. Only Nvidia GPUs with compute capability of at least 7.0 run quickly with mixed_float16.
If you will use compatible GPU(s) not attached to this host, e.g. by running a multi-worker model, you can ignore this warning. This message will only be logged once
Mixed precision: ON


2026-04-22 14:21:55.772534: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Total species : 234 | Training files : 35549
After quality filter : 35549 samples
primary_label
rubthr1    499
banana     498
fepowl     497
soulap1    497
houspa     496
Name: count, dtype: int64
Soundscape segments : 1478
fold
0    7110
1    7110
2    7110
3    7110
4    7109
Name: count, dtype: int64


In [2]:
WEIGHTS_FILE = "/kaggle/input/models/tensorflow/efficientnet/tensorflow2/b2-feature-vector/1"

In [9]:
import tensorflow_hub as hub

In [10]:
# 8. MODEL
def build_model(num_classes=NUM_CLASSES):
    inputs   = keras.Input(shape=CFG.IMG_SHAPE, name="spectrogram")
    backbone = hub.KerasLayer(WEIGHTS_FILE,
                   trainable=True,
                             name="backbone")
    x = layers.Lambda(lambda x: tf.cast(x, tf.float32))(inputs)
    x = backbone(x)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(CFG.DROP_RATE)(x)
    x = layers.Dense(512)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("swish")(x)
    x = layers.Dropout(CFG.DROP_RATE2)(x)
    outputs = layers.Dense(num_classes, activation="sigmoid", dtype="float32")(x)
    return keras.Model(inputs, outputs, name="BirdCLEF_EfficientNetV2B2")

In [11]:
# 9. FOCAL LOSS & LR SCHEDULE
def focal_loss(gamma=2.0, alpha=0.25):
    def _loss(y_true, y_pred):
        y_true = tf.cast(y_true, tf.float32)
        y_pred = tf.clip_by_value(tf.cast(y_pred, tf.float32),
                                  tf.keras.backend.epsilon(), 1 - tf.keras.backend.epsilon())
        bce   = -(y_true*tf.math.log(y_pred) + (1-y_true)*tf.math.log(1-y_pred))
        p_t   = y_true*y_pred + (1-y_true)*(1-y_pred)
        a_t   = y_true*alpha  + (1-y_true)*(1-alpha)
        return tf.reduce_mean(a_t * tf.pow(1.0-p_t, gamma) * bce)
    return _loss

In [12]:
class WarmupCosineDecay(keras.optimizers.schedules.LearningRateSchedule):
    def __init__(self, base_lr, min_lr, total_steps, warmup_steps):
        super().__init__()
        self.base_lr, self.min_lr = float(base_lr), float(min_lr)
        self.total_steps, self.warmup_steps = float(total_steps), float(warmup_steps)

    def __call__(self, step):
        step      = tf.cast(step, tf.float32)
        warmup_lr = self.base_lr * step / self.warmup_steps
        progress  = (step - self.warmup_steps) / (self.total_steps - self.warmup_steps)
        cosine_lr = self.min_lr + 0.5 * (self.base_lr - self.min_lr) * (1.0 + tf.cos(math.pi * progress))
        return tf.where(step < self.warmup_steps, warmup_lr, cosine_lr)

    def get_config(self):
        return dict(base_lr=self.base_lr, min_lr=self.min_lr,
                    total_steps=self.total_steps, warmup_steps=self.warmup_steps)

In [13]:
# 10. TRAINING LOOP
def train_fold(fold):
    print(f"\nFOLD {fold}")
    fold_train = train_df[train_df["fold"] != fold].copy()
    fold_val   = train_df[train_df["fold"] == fold].copy()
    if len(sc_train_df) > 0:
        sc_copy = sc_train_df.copy(); sc_copy["fold"] = -1
        sc_copy["secondary_labels"] = sc_copy.get("secondary_labels", pd.Series([""]*len(sc_copy)))
        fold_train = pd.concat([fold_train, sc_copy], ignore_index=True)
    print(f"  Train: {len(fold_train)} | Val: {len(fold_val)}")

    train_ds = build_dataset(fold_train, training=True)
    val_ds   = build_dataset(fold_val, training=False)

    steps_per_epoch = max(1, len(fold_train) // CFG.BATCH_SIZE)
    lr_schedule = WarmupCosineDecay(
        CFG.LEARNING_RATE, CFG.MIN_LR,
        total_steps=steps_per_epoch*CFG.EPOCHS,
        warmup_steps=steps_per_epoch*CFG.WARMUP_EPOCHS,
    )

    model = build_model()
    model.compile(
        optimizer=keras.optimizers.AdamW(learning_rate=lr_schedule, weight_decay=CFG.WEIGHT_DECAY),
        loss=focal_loss(2.0, 0.25),
        metrics=[keras.metrics.AUC(curve="ROC", multi_label=True, name="roc_auc")],
    )
    if fold == CFG.TRAIN_FOLDS[0]: model.summary(line_length=80)

    ckpt_path = f"model_fold{fold}.keras"
    callbacks = [
        keras.callbacks.ModelCheckpoint(ckpt_path, monitor="val_roc_auc", mode="max",
                                        save_best_only=True, verbose=1),
        keras.callbacks.EarlyStopping(monitor="val_roc_auc", mode="max", patience=CFG.PATIENCE,
                                      restore_best_weights=True, verbose=1),
        keras.callbacks.CSVLogger(f"history_fold{fold}.csv"),
    ]

    history = model.fit(train_ds, validation_data=val_ds, epochs=CFG.EPOCHS,
                        callbacks=callbacks, verbose=1)
    best_auc = max(history.history.get("val_roc_auc", [0.0]))
    print(f"  Fold {fold} best val ROC-AUC: {best_auc:.4f}")
    return model, history

In [14]:
trained_models, fold_histories = [], []
for fold in CFG.TRAIN_FOLDS:
    model, history = train_fold(fold)
    trained_models.append(model); fold_histories.append(history)
    gc.collect()
print("Training complete.")


FOLD 0
  Train: 29917 | Val: 7110


ValueError: Exception encountered when calling layer "backbone" (type KerasLayer).

in user code:

    File "/usr/local/lib/python3.12/dist-packages/tensorflow_hub/keras_layer.py", line 250, in call  *
        result = smart_cond.smart_cond(training,

    ValueError: Could not find matching concrete function to call loaded from the SavedModel. Got:
      Positional arguments (3 total):
        * <tf.Tensor 'inputs:0' shape=(None, 128, 384, 3) dtype=float16>
        * False
        * None
      Keyword arguments: {}
    
     Expected these arguments to match one of the following 4 option(s):
    
    Option 1:
      Positional arguments (3 total):
        * TensorSpec(shape=(None, None, None, 3), dtype=tf.float32, name='image_input')
        * True
        * None
      Keyword arguments: {}
    
    Option 2:
      Positional arguments (3 total):
        * TensorSpec(shape=(None, None, None, 3), dtype=tf.float32, name='inputs')
        * True
        * None
      Keyword arguments: {}


In [ ]:
# 11. SOUNDSCAPE INFERENCE
def load_soundscape_chunks(filepath, duration=CFG.SOUNDSCAPE_DURATION,
                            chunk_dur=float(CFG.DURATION), overlap=CFG.INFER_OVERLAP):
    try:
        audio, _ = librosa.load(filepath, sr=CFG.SAMPLE_RATE, duration=duration, mono=True)
        audio = audio.astype(np.float32)
    except:
        audio = np.zeros(int(duration*CFG.SAMPLE_RATE), dtype=np.float32)

    step, chunks, start = chunk_dur*(1.0-overlap), [], 0.0
    while round(start+chunk_dur, 6) <= duration + 1e-6:
        s     = int(start*CFG.SAMPLE_RATE)
        chunk = audio[s:s+CFG.N_SAMPLES]
        if len(chunk) < CFG.N_SAMPLES: chunk = np.pad(chunk, (0, CFG.N_SAMPLES-len(chunk)))
        chunks.append((int(round(start+chunk_dur)), chunk)); start += step
    return chunks

def predict_soundscape(filepath, models):
    filename  = Path(filepath).stem
    chunks    = load_soundscape_chunks(filepath)
    waveforms = np.stack([w for _, w in chunks])
    end_secs  = [e for e, _ in chunks]
    images    = tf.vectorized_map(waveform_to_image, tf.constant(waveforms))
    all_preds = []
    for m in models:
        preds = np.concatenate([m.predict(images[i:i+CFG.INFER_BATCH_SIZE], verbose=0)
                                for i in range(0, len(images), CFG.INFER_BATCH_SIZE)])
        all_preds.append(preds)
    avg = np.mean(all_preds, axis=0)
    return [(f"{filename}_{e}", avg[i]) for i, e in enumerate(end_secs)]

def run_inference(models, test_dir=CFG.TEST_SOUNDSCAPES_DIR):
    test_files = sorted(Path(test_dir).glob("*.ogg"))
    if not test_files:
        print(f"No .ogg files in {test_dir} (expected at submission time)"); return []
    print(f"Running inference on {len(test_files)} soundscapes...")
    all_results = []
    for i, fp in enumerate(test_files, 1):
        all_results.extend(predict_soundscape(str(fp), models))
        if i % 50 == 0 or i == len(test_files):
            print(f"  [{i}/{len(test_files)}] {fp.name}")
    return all_results

# 12. BUILD SUBMISSION
def build_submission(results):
    sample_sub = pd.read_csv(CFG.SAMPLE_SUB_CSV)
    if not results:
        print("No predictions — returning zeroed submission."); return sample_sub
    pred_dict = dict(results)
    sub_rows, missing = [], 0
    for row_id in sample_sub["row_id"]:
        if row_id in pred_dict: sub_rows.append(pred_dict[row_id])
        else: sub_rows.append(np.zeros(NUM_CLASSES, dtype=np.float32)); missing += 1
    if missing: print(f"{missing} row_ids had no prediction (filled with 0).")
    sub_df = pd.DataFrame(np.stack(sub_rows), columns=ALL_SPECIES)
    sub_df.insert(0, "row_id", sample_sub["row_id"])
    return sub_df

results    = run_inference(trained_models)
submission = build_submission(results)
submission.to_csv("submission.csv", index=False)
print(f"submission.csv saved — shape: {submission.shape}")
print(submission.head(3).to_string())